# Setup
Creates the Delta tables, grants the service principal the privileges Zerobus requires and resets the previous run.

Zerobus does not create tables and a target must not be dropped or recreated while it is in use, so the tables are created once and only emptied here.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("zerobus_client_id", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
client_id = dbutils.widgets.get("zerobus_client_id")
target = f"{catalog}.{schema}"

In [ ]:
columns = """
    event_id STRING NOT NULL,
    producer_id STRING NOT NULL,
    seq BIGINT NOT NULL,
    order_id BIGINT NOT NULL,
    event_type STRING NOT NULL,
    amount DOUBLE,
    event_time TIMESTAMP NOT NULL
"""

spark.sql(f"CREATE TABLE IF NOT EXISTS {target}.events_bronze ({columns}) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {target}.events_silver ({columns}) USING DELTA")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target}.producer_metrics (
        producer_id STRING, step STRING, from_seq BIGINT, to_seq BIGINT, records BIGINT,
        durable_seconds DOUBLE, records_per_second DOUBLE, queryable_seconds DOUBLE, finished_at TIMESTAMP
    ) USING DELTA
""")

for table in ("events_bronze", "events_silver", "producer_metrics"):
    spark.sql(f"DELETE FROM {target}.{table}")
dbutils.fs.rm(f"/Volumes/{catalog}/{schema}/checkpoints/silver", True)

In [ ]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{client_id}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {target} TO `{client_id}`")
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {target}.events_bronze TO `{client_id}`")